In [ ]:
import json
from itertools import islice
import pandas as pd
import re
from langdetect import DetectorFactory, detect, detect_langs
from langdetect.lang_detect_exception import LangDetectException
import emot
import underthesea
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

In [ ]:
with open('data.json', 'r', encoding='utf-8') as f:
    json_data = json.load(f)
comment_data = dict(islice(json_data.items(), 3, None))

In [ ]:
df = pd.DataFrame(comment_data)
df = df.transpose()
df

# Tiền xử lý

In [ ]:
df.isna().sum()

In [ ]:
# lọc ra các comment là tiếng Việt
def is_vietnamese(text):
    langs = detect_langs(text)
    for i in langs:
        if i.lang=='vi' and i.prob>=0.9:
            return True
    return False

df = df[df['noi_dung_comment'].apply(is_vietnamese)]

for text in df['noi_dung_comment']:
    print(text)
    print('\n')

In [ ]:
# chuyển về từ viết thường
df['noi_dung_comment'] = df['noi_dung_comment'].str.lower()
for text in df['noi_dung_comment']:
    print(text)
    print('\n')

In [ ]:
# xóa dấu câu và chữ số
def remove_punctuation(text):
    s = re.sub(r'[\W\d_]+', ' ', text)
    return s

df['noi_dung_comment'] = df['noi_dung_comment'].apply(remove_punctuation)
for text in df['noi_dung_comment']:
    print(text)
    print('\n')


In [ ]:
# chuyển đổi emoji, icon

emot_obj = emot.core.emot()

def convert_emotion(text):
    emoji = emot_obj.emoji(text)
    emoticons = emot_obj.emoticons(text)
    
    for i,j in zip(emoji['value'], emoji['mean']):
        text = text.replace(i,j.replace(':', ' '))
        
    for i,j in zip(emoticons['value'], emoticons['mean']):
        text = text.replace(i,'_'.join(tmp for tmp in j.split()))
    
    return text

df['noi_dung_comment'] = df['noi_dung_comment'].apply(convert_emotion)
for text in df['noi_dung_comment']:
    print(text)
    print('\n')

In [ ]:
# chuẩn hóa từ viết tắt
d = {
    'ksan':'khách_sạn',
    'xquanh':'xung quang',
    'ok':'ổn'
}

def normalize(text):
    words = text.split()
    return " ".join([d.get(word, word) for word in words])

df['noi_dung_comment'] = df['noi_dung_comment'].apply(normalize)

for text in df['noi_dung_comment']:
    print(text)
    print('\n')

In [ ]:
def tokenize(text):
    return underthesea.word_tokenize(text)

df['tokenize'] = df['noi_dung_comment'].apply(tokenize)
df[['noi_dung_comment', 'tokenize']]

In [ ]:
# xóa stop words

def remove_stop_words(list):
    with open('Data/vietnamese-stopwords.txt', 'r', encoding='utf-8') as f:
        stop_words = set(f.read().split('\n'))
    
    return [i for i in list if (i not in stop_words) and (i not in ('không', 'chưa', 'chẳng'))]

df['tokenize'] = df['tokenize'].apply(remove_stop_words)

df['tokenize'] = df['tokenize'].apply(
    lambda words: [w.replace(' ', '_') for w in words]
)

df[['noi_dung_comment', 'tokenize']]

# EDA

In [ ]:
# word cloud 20 từ xuất hiện nhiều nhất
all_words = df['tokenize'].explode().dropna().tolist()

word_counts = Counter(all_words)
top20 = pd.DataFrame(word_counts.most_common(20), columns=['tu_vung', 'tan_suat'])

plt.figure(figsize=(10,20))
sns.barplot(data=top20, x='tan_suat', y='tu_vung', hue='tu_vung')
plt.title('Top 20 từ vựng xuất hiện nhiều nhất')
plt.xlabel('Tần suất')
plt.ylabel('Từ vựng')
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
wc = WordCloud(
    width=800,
    height=400,
    background_color='white',
    colormap='plasma'
).generate_from_frequencies(word_counts)


plt.figure(figsize=(12,6))
plt.imshow(wc, interpolation="bilinear")
plt.axis('off')
plt.title('Word cloud tổng thể')
plt.show()

In [ ]:
groups = ['Cặp đôi', 'Gia đình có em bé']
aspects = {
    'hồ bơi': 'hồ_bơi|bể_bơi|pool',
    'trẻ em':'trẻ_em|em_bé',
    'bữa sáng':'bữa_sáng|ăn_sáng',
    'lãng mạn':'lãng_mạn',
    'riêng tư':'riêng_tư|yên_tĩnh|private'
}

sub_df = df[df['loai_hinh_du_lich'].isin(groups)].copy()
sub_df['tokens'] = sub_df['tokenize'].apply(lambda x: ' '.join(x))

stats = pd.DataFrame({asp:sub_df['tokens'].str.count(pat) for asp, pat in aspects.items()})
stats['Nhóm'] = sub_df['loai_hinh_du_lich']

stats.groupby('Nhóm').sum().T.plot(kind='bar', figsize=(9,4), rot=0, grid=True)
plt.title('So sánh nhóm Gia đình có em bé và Cặp đôi')
plt.ylabel('Số lần xuất hiện')
plt.tight_layout()
plt.show()